In [95]:
import pandas as pd
import numpy as np 



In [65]:
df=pd.read_csv("raw_car_dataset.csv")

In [ ]:
print(df.head(10))

In [67]:
print(df.shape)

(145, 6)


In [68]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    str    
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    str    
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 6.9 KB
None


In [69]:
print("missing values:")
print(df.isnull().sum())

missing values:
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [70]:
print("show the counts of location:")
print(df["Location"].value_counts(dropna=False))

show the counts of location:
Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


In [71]:
print(" clean section mareyna $ iyo ,")
df["Price"]=df["Price"].replace(r"[\$,]","", regex=True).astype(float)
print(df["Price"].head(10))

 clean section mareyna $ iyo ,
0    1500.0
1    4171.0
2    5331.0
3    1500.0
4    1500.0
5    1500.0
6    1500.0
7    1500.0
8    2291.0
9    1500.0
Name: Price, dtype: float64


In [72]:
print(df["Price"].skew())


7.871113660850301


3-Fix CATEGORY LOCATION

In [73]:

df["Location"]=df["Location"].replace({"Subrb": "Suburb", "??": pd.NA})
print("laction counts after fixed type/unknown")
print(df["Location"].value_counts(dropna=False))

laction counts after fixed type/unknown
Location
City      59
Suburb    53
Rural     21
NaN       12
Name: count, dtype: int64


4-Impute Missing Values (justify choices)


In [74]:
df["Odometer_km"]=df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"]=df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"]=df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"]=df["Location"].fillna(df["Location"].mode()[0])
print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


5-Remove Duplicates

In [75]:
bofore=df.shape

In [76]:
df=df.drop_duplicates()

In [77]:
after=df.shape

In [78]:
print("before: ", bofore,"After: ",after)

before:  (145, 6) After:  (140, 6)


7-Outliers (IQR capping)

In [79]:
def iqr_fun(series,k=1.5):
    q1,q3=series.quantile([0.25,0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
low_price, high_price=iqr_fun(df["Price"])
low_odometer, high_odometer=iqr_fun(df["Odometer_km"])

In [80]:
print("IQR Of Price : " )
print("Low_Price: ", low_price , "High_Price: " , high_price)

IQR Of Price : 
Low_Price:  -2984.625 High_Price:  8974.375


In [81]:
print("IQR Of Odometer_km : " )
print("Low_Odometer: ", low_odometer , "High_Odometer: " , high_odometer)

IQR Of Odometer_km : 
Low_Odometer:  -6642.25 High_Odometer:  271987.75


after clip

In [82]:
df.loc[:,"Price"] = df ["Price"].clip(lower = low_price , upper = high_price)
df.loc[:,"Odometer_km"] = df ["Odometer_km"].clip(lower = low_odometer , upper = high_odometer)



In [83]:
print("The Price after IQR Clipping :")
print(df["Price"].describe())


The Price after IQR Clipping :
count     140.000000
mean     3175.456250
std      2601.848629
min      1500.000000
25%      1500.000000
50%      1500.000000
75%      4489.750000
max      8974.375000
Name: Price, dtype: float64


In [84]:
print("The Odometer_km after IQR Clipping :")
print(df["Odometer_km"].describe())

The Odometer_km after IQR Clipping :
count       140.000000
mean     130945.403571
std       53815.006935
min        5000.000000
25%       97844.000000
50%      128548.000000
75%      167501.500000
max      271987.750000
Name: Odometer_km, dtype: float64


7-One-Hot Encode Categorical(s)

In [85]:


df = pd.get_dummies(
    df,
    columns=["Location"],
    prefix="Location",
    dtype=int
)

dummy_cols = [c for c in df.columns if c.startswith("Location_")]

print("\nSTEP 7: ONE-HOT ENCODING")
print("New dummy columns created:")
print(dummy_cols)


STEP 7: ONE-HOT ENCODING
New dummy columns created:
['Location_City', 'Location_Rural', 'Location_Suburb']


8-Feature Engineering (no leakage)

In [86]:
CURRENT_YEAR=2026
df["CarAge"]= CURRENT_YEAR-df["Year"]
df["Km_per_year"]= (df["Odometer_km"] / df["CarAge"] + 1)


In [87]:
df["Is_Urban"] = df["Location_Urban"].astype(int) if "Location_Urban" in df.columns else 0

df["LogPrice"] = np.log1p(df["Price"])

In [88]:

print(df[["CarAge", "Km_per_year", "Is_Urban", "LogPrice"]].head())

   CarAge   Km_per_year  Is_Urban  LogPrice
0      28   4923.500000         0  7.313887
1      10  12855.800000         0  8.336151
2      12   8942.833333         0  8.581482
3      27   5254.259259         0  7.313887
4       4  32138.000000         0  7.313887


9-Feature Scaling (X only)

In [89]:
from sklearn.preprocessing import StandardScaler

In [90]:
from sklearn.preprocessing import StandardScaler

dont_scale = {'Price', 'LogPrice'}
exclude = [c for c in df.columns if c.startswith('Location_')] + ['Is_Urban']
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in dont_scale and c not in exclude]

df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])
df[scale_cols].head()

,Odometer_km,Doors,Accidents,Year,CarAge,Km_per_year
0,0.128390,0.254091,0.316968,-1.686714,1.686714,-0.615631
1,-0.044709,0.254091,-0.820867,0.794617,-0.794617,0.070446
2,-0.440923,0.254091,-0.820867,0.518913,-0.518913,-0.267993
3,0.203135,0.254091,0.316968,-1.548862,1.548862,-0.587024
4,-0.044709,-0.931668,-0.820867,1.621727,-1.621727,1.738196


10-Final Checks & Save

In [91]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_City    140 non-null    int64  
 6   Location_Rural   140 non-null    int64  
 7   Location_Suburb  140 non-null    int64  
 8   CarAge           140 non-null    float64
 9   Km_per_year      140 non-null    float64
 10  Is_Urban         140 non-null    int64  
 11  LogPrice         140 non-null    float64
dtypes: float64(8), int64(4)
memory usage: 13.3 KB
None


In [92]:
print("missing values:")
print(df.isnull().sum())

missing values:
Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0
Location_Rural     0
Location_Suburb    0
CarAge             0
Km_per_year        0
Is_Urban           0
LogPrice           0
dtype: int64


In [93]:
print("Describe ")
print(df.describe())

Describe 
             Price   Odometer_km         Doors     Accidents          Year  \
count   140.000000  1.400000e+02  1.400000e+02  1.400000e+02  1.400000e+02   
mean   3175.456250  3.172066e-18  2.077703e-16  6.344132e-18  2.686740e-15   
std    2601.848629  1.003591e+00  1.003591e+00  1.003591e+00  1.003591e+00   
min    1500.000000 -2.348743e+00 -2.117428e+00 -8.208670e-01 -1.686714e+00   
25%    1500.000000 -6.173048e-01 -9.316683e-01 -8.208670e-01 -8.596039e-01   
50%    1500.000000 -4.470894e-02  2.540913e-01 -8.208670e-01 -3.249362e-02   
75%    4489.750000  6.817310e-01  2.540913e-01  3.169684e-01  8.290796e-01   
max    8974.375000  2.630285e+00  1.439851e+00  2.592639e+00  1.759579e+00   

       Location_City  Location_Rural  Location_Suburb        CarAge  \
count     140.000000      140.000000       140.000000  1.400000e+02   
mean        0.485714        0.150000         0.364286 -9.516197e-18   
std         0.501590        0.358354         0.482957  1.003591e+00   
min

In [94]:
OUT_PATH="car_l3_clean_ready.csv"
df.to_csv(OUT_PATH)
print("Saved to: " , OUT_PATH)

Saved to:  car_l3_clean_ready.csv
